# 03 · MTrans teacher (multi-class, 500 labelled KITTI frames)

Trains the vendored MTrans (`ocgf/teachers/mtrans/`, MPL-2.0) for car, large vehicle, pedestrian and cyclist:

- **labelled objects**: human 3D labels of the 500 frames in `configs/splits/kitti_object/mtrans_labelled_500.txt` only;
- **unlabelled objects**: zero-shot detector boxes (dataset `ocgf-detector2d-v1`) on the other KITTI train frames; they feed only the point-reconstruction loss;
- **KITTI val**: evaluation only (ground-truth 2D boxes as input, as MTrans reports it). It never selects a checkpoint: the last epoch is the result.

A checkpoint is saved after every epoch. A session stops itself before the 12 h limit; to continue, attach this notebook's previous output (**Add Input → Your Work → Notebooks**) and run again with the same `EXPERIMENT`.

## Before you run

1. **Settings → Accelerator:** GPU T4 x2 (one is used). **Internet:** On (code clone).
2. **Add Input:** `garymk/kitti-3d-object-detection-dataset` and your `ocgf-detector2d-v1`.
3. **Save Version → Save & Run All.** The smoke test runs first (about 2 min) and stops the run early if anything is broken.

In [ ]:
from pathlib import Path
import time

T0 = time.time()                     # the time budget is measured from here
REPO_URL   = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF    = "p0/skeleton-geometry"  # branch, tag or commit (main once merged); pin a commit for the final run
EXPERIMENT = "mtrans_v1"             # output folder; keep it the same across sessions to resume
CONFIG     = "configs/teachers/mtrans_v1.yaml"
LABELLED_FRAMES = None               # None: the config's 500-frame list; S1 ablation: "configs/splits/kitti_object/mtrans_labelled_125.txt"
EPOCHS     = None                    # None: the config (100); a resumed run with another count re-plans its learning rate
RUN_SMOKE_TEST    = True
RUN_TRAINING      = True
RUN_VAL_INFERENCE = True             # once all epochs are done: detector boxes on KITTI val -> object file
TIME_BUDGET_HOURS = 11.0             # stop cleanly this long after the notebook started (a session ends at 12 h)
AMP        = False                   # mixed precision (fp16 forward, fp32 loss): faster on a T4; the original trains in fp32
THREADS    = 32                      # parallel readers: /kaggle/input is slow per file but serves parallel reads

INPUT_ROOT = Path("/kaggle/input")
WORKING    = Path("/kaggle/working")
OUT        = WORKING / EXPERIMENT
CACHE      = Path("/kaggle/temp/mtrans_cache")      # per-frame point cache, rebuilt each session
STAGE_DIR  = Path("/kaggle/temp/kitti")             # local copy of KITTI images, calibration and labels
CODE_DIR   = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

import torch
from ocgf.utils.provenance import environment
print(environment())
assert torch.cuda.is_available(), "No GPU: choose Settings > Accelerator > GPU T4 x2 and restart the session."
print("GPU:", torch.cuda.get_device_name(0))

## Inputs

KITTI's `training` folders, the detector's KITTI file, and the previous session's checkpoint if its output is attached.

In [ ]:
import json
from ocgf.datasets.locate import find_kitti_object

kitti = find_kitti_object(INPUT_ROOT)
for name in ("image_2", "velodyne", "calib", "label_2"):
    print(f"{name:>9}: {kitti.get(name, 'NOT FOUND')}")
assert len(kitti) == 4, "attach garymk/kitti-3d-object-detection-dataset (Add Input)"

detections = sorted(INPUT_ROOT.rglob("detector2d_v1/kitti/training.txt"))
assert detections, "attach the ocgf-detector2d-v1 dataset (Add Input): it holds detector2d_v1/kitti/training.txt"
DETECTIONS = detections[0]
print("detections:", DETECTIONS)

def epochs_done(ckpt_dir):
    p = ckpt_dir / "progress.json"
    return json.loads(p.read_text())["epoch"] + 1 if p.exists() else 0

if not (OUT / "ckpt" / "last.pt").exists():
    previous = [p.parent.parent for p in INPUT_ROOT.rglob(f"{EXPERIMENT}/ckpt/last.pt")]
    if previous:
        source = max(previous, key=lambda d: epochs_done(d / "ckpt"))
        shutil.copytree(source, OUT, dirs_exist_ok=True)        # checkpoint, records, metrics of the last session
        print(f"continuing from {source}: {epochs_done(OUT / 'ckpt')} epochs done")
print("starting point:", f"{epochs_done(OUT / 'ckpt')} epochs done" if (OUT / "ckpt").exists() else "a fresh run")

args = (f'--input-root "{INPUT_ROOT}" --detections "{DETECTIONS}" --cache "{CACHE}" --config "{CODE_DIR / CONFIG}" '
        f'--stage-dir "{STAGE_DIR}" --threads {THREADS}' + (" --amp" if AMP else ""))
if LABELLED_FRAMES:
    args += f" --labelled-frames {LABELLED_FRAMES}"
if EPOCHS:
    args += f" --epochs {EPOCHS}"
print("arguments:", args)

## Smoke test

First every session copies KITTI's images, calibration and labels to local disk and caches each frame's in-image LiDAR points, with `THREADS` parallel readers (files under `/kaggle/input` are slow to read one at a time). The first session also builds the object records; later sessions reuse them from the attached output. Then one epoch on 64 objects per pool, an evaluation and a checkpoint reload: a broken pipeline fails here, not hours into the run.

In [ ]:
if RUN_SMOKE_TEST:
    sh(f'{sys.executable} scripts/train_mtrans.py {args} --out "{OUT}" --smoke 64', cwd=CODE_DIR)
    smoke_dir = OUT.with_name(OUT.name + "_smoke")
    OUT.mkdir(parents=True, exist_ok=True)
    if not (OUT / "records.json").exists():
        shutil.copyfile(smoke_dir / "records.json", OUT / "records.json")    # full pools: the real run skips the build
    shutil.rmtree(smoke_dir)            # its checkpoint was only a test

## Train

The log line shows running averages (loss, mean 3D IoU of the labelled boxes, point-reconstruction error) and, per step, the time spent waiting for data and computing. Every `eval_every` epochs and at the end, KITTI val per class: mean IoU and recall at IoU 0.7. The same lines go to `train.log` in the output folder.

In [ ]:
if RUN_TRAINING:
    OUT.mkdir(parents=True, exist_ok=True)
    hours_left = TIME_BUDGET_HOURS - (time.time() - T0) / 3600
    assert hours_left > 0.25, "the time budget is used up: start a new session"
    sh(f'{sys.executable} scripts/train_mtrans.py {args} --out "{OUT}" --time-budget-hours {hours_left:.2f}', cwd=CODE_DIR)

## Progress

In [ ]:
progress = json.loads((OUT / "ckpt" / "progress.json").read_text()) if (OUT / "ckpt" / "progress.json").exists() else None
FINISHED = bool(progress) and progress["epoch"] + 1 >= progress["epochs"]
if progress:
    print(f"{progress['epoch'] + 1} of {progress['epochs']} epochs done" + ("" if FINISHED else "; run again to continue"))
    evals = [e for e in map(json.loads, (OUT / "metrics.jsonl").read_text().splitlines()) if "val" in e]
    if evals:
        last = evals[-1]
        print(f"\nKITTI val after epoch {last['epoch']} (ground-truth 2D boxes as input):")
        print(f"{'class':>14} {'objects':>8} {'mean IoU':>9} {'R@0.5':>7} {'R@0.7':>7} {'|conf-IoU|':>11}")
        for cls, v in last["val"].items():
            if v["n"]:
                print(f"{cls:>14} {v['n']:>8} {v['mean_iou3d']:>9.3f} {v['recall_0.5']:>7.3f} {v['recall_0.7']:>7.3f} {v['conf_abs_err']:>11.3f}")
else:
    print("no checkpoint yet")

## KITTI val with detector boxes

Once training is finished: the checkpoint on the detector's boxes (score ≥ 0.1) for the val frames, the setting MTrans meets at inference. The object file is the input for the teacher-quality evaluation (matching to ground truth comes with the evaluation step).

In [ ]:
if RUN_VAL_INFERENCE and FINISHED:
    sh(f'{sys.executable} scripts/run_mtrans.py --checkpoint "{OUT / "ckpt" / "last.pt"}" --input-root "{INPUT_ROOT}" '
       f'--detections "{DETECTIONS}" --kitti-split val --cache "{CACHE}" --stage-dir "{STAGE_DIR}" --threads {THREADS} --out "{OUT / "objects" / "kitti" / "val.jsonl"}"',
       cwd=CODE_DIR)
elif RUN_VAL_INFERENCE:
    print("runs after the last epoch")
sh(f'du -sh "{OUT}"/* | sort -h')

## Output (`/kaggle/working/<EXPERIMENT>`)

| Path | Content |
|---|---|
| `ckpt/last.pt` | model (with class anchors), optimizer, scheduler, configs; what the next session resumes from and what inference uses |
| `ckpt/progress.json` | epochs done |
| `records.json` | object pools (labelled / unlabelled / val) and the class anchors |
| `metrics.jsonl` | one line per epoch: training averages, and KITTI val per class where evaluated |
| `val_objects.jsonl` | after the last epoch: per-object 3D IoU and confidence on KITTI val |
| `objects/kitti/val.jsonl` | detector boxes on KITTI val -> 3D boxes with raw confidence (+ `.manifest.json`) |
| `manifest.json` | code commit, config, label-list and detections hashes, counts, anchors |

When all epochs are done, save the output as the private dataset `ocgf-mtrans-v1` (Output → New Dataset).